<a href="https://colab.research.google.com/github/Basmala135/grad_project/blob/main/grad_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, StratifiedGroupKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve, auc
)
import matplotlib.pyplot as plt

In [17]:
DRIVE_ROOT      = Path('/content/drive/MyDrive/Colab Notebooks')
BOUBEZOUL_DIR   = DRIVE_ROOT / 'fall_data'        # folder with Boubezoul CSVs
NORMAL_CSV_PATH = DRIVE_ROOT / 'anomaly_data.csv' # your normal riding CSV

SAMPLE_RATE   = 100
WINDOW_SEC    = 2.0
OVERLAP       = 0.5
WINDOW_SIZE   = int(SAMPLE_RATE * WINDOW_SEC)     # 200
STEP_SIZE     = int(WINDOW_SIZE * (1 - OVERLAP))  # 100
CHANNELS      = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']

print(f"Boubezoul dir exists : {BOUBEZOUL_DIR.exists()}")
print(f"Normal CSV exists    : {NORMAL_CSV_PATH.exists()}")

Boubezoul dir exists : True
Normal CSV exists    : True


In [18]:
BOUBEZOUL_COLS = {
    'Ax(m/s²)': 'ax', 'Ay(m/s²)': 'ay', 'Az(m/s²)': 'az',
    'Rx(°/s)': 'gx', 'Ry.(°/s)': 'gy', 'Rz(°/s)': 'gz',
}

def load_accident_files(folder: Path, col_map: dict):
    """
    Returns a list of (dataframe, group_id) tuples.
    group_id = filename, so windows from the same trial never split
    across train/test.
    """
    results = []
    csv_files = sorted(folder.glob("*.csv"))
    print(f"Found {len(csv_files)} accident CSV files")
    for f in csv_files:
        try:
            df = pd.read_csv(f, sep='\t', encoding='latin1')
        except Exception:
            df = pd.read_csv(f, encoding='latin1')
        df = df.rename(columns=col_map)
        missing = [c for c in CHANNELS if c not in df.columns]
        if missing:
            print(f"  SKIP {f.name}: missing columns {missing}")
            continue
        df = df[CHANNELS].dropna().reset_index(drop=True)
        if len(df) < WINDOW_SIZE:
            print(f"  SKIP {f.name}: only {len(df)} rows, need {WINDOW_SIZE}")
            continue
        results.append((df, f.stem))   # group_id = filename without extension
        print(f"  Loaded {f.name}: {len(df)} rows")
    return results

accident_files = load_accident_files(BOUBEZOUL_DIR, BOUBEZOUL_COLS)
print(f"\nTotal usable accident trials: {len(accident_files)}")

Found 7 accident CSV files


/tmp/ipykernel_3416/3489835518.py:17: DtypeWarning: Columns (1,2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, sep='\t', encoding='latin1')


  Loaded Fall Like manoeuvre 1.csv: 132478 rows


/tmp/ipykernel_3416/3489835518.py:17: DtypeWarning: Columns (1,2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, sep='\t', encoding='latin1')


  Loaded Fall Like manoeuvre 2.csv: 148926 rows


/tmp/ipykernel_3416/3489835518.py:17: DtypeWarning: Columns (1,2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, sep='\t', encoding='latin1')
/tmp/ipykernel_3416/3489835518.py:17: DtypeWarning: Columns (1,2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, sep='\t', encoding='latin1')


  Loaded Fall Like manoeuvre 3.csv: 128446 rows
  Loaded Fall in a curve.csv: 78461 rows


/tmp/ipykernel_3416/3489835518.py:17: DtypeWarning: Columns (1,2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, sep='\t', encoding='latin1')
/tmp/ipykernel_3416/3489835518.py:17: DtypeWarning: Columns (1,2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, sep='\t', encoding='latin1')


  Loaded Fall in the roundabout.csv: 66365 rows
  Loaded Fall on a slippery straight road section.csv: 74045 rows
  Loaded Fall with leaning of the motorcycle.csv: 81663 rows

Total usable accident trials: 7


/tmp/ipykernel_3416/3489835518.py:17: DtypeWarning: Columns (1,2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, sep='\t', encoding='latin1')


In [19]:
df_normal_full = pd.read_csv(NORMAL_CSV_PATH)
NORMAL_CHANNELS_MAP = {
    'Acc_X': 'ax', 'Acc_Y': 'ay', 'Acc_Z': 'az',
    'Gyr_X': 'gx', 'Gyr_Y': 'gy', 'Gyr_Z': 'gz',
}
df_normal_full = df_normal_full.rename(columns=NORMAL_CHANNELS_MAP)

N_NORMAL_CHUNKS = 40  # split into 40 pseudo-trials
chunk_size = len(df_normal_full) // N_NORMAL_CHUNKS

normal_files = []
for i in range(N_NORMAL_CHUNKS):
    start = i * chunk_size
    end   = start + chunk_size if i < N_NORMAL_CHUNKS - 1 else len(df_normal_full)
    chunk = df_normal_full[CHANNELS].iloc[start:end].reset_index(drop=True)
    normal_files.append((chunk, f"normal_chunk_{i}"))

print(f"Normal data split into {len(normal_files)} pseudo-trial chunks "
      f"(~{chunk_size} rows each)")

Normal data split into 40 pseudo-trial chunks (~83622 rows each)


In [20]:
def make_windows_grouped(file_list, label: int):
    """
    Returns X (windows), y (labels), groups (trial id per window).
    """
    X, y, groups = [], [], []
    for df, group_id in file_list:
        data = df.values.astype(np.float32)
        for start in range(0, len(data) - WINDOW_SIZE + 1, STEP_SIZE):
            X.append(data[start:start + WINDOW_SIZE])
            y.append(label)
            groups.append(group_id)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int8), np.array(groups)

X_acc, y_acc, groups_acc = make_windows_grouped(accident_files, label=1)
X_nor, y_nor, groups_nor = make_windows_grouped(normal_files,   label=0)

print(f"\nAccident windows: {len(X_acc)}  from {len(set(groups_acc))} trials")
print(f"Normal windows  : {len(X_nor)}  from {len(set(groups_nor))} trials")

X      = np.concatenate([X_acc, X_nor], axis=0)
y      = np.concatenate([y_acc, y_nor], axis=0)
groups = np.concatenate([groups_acc, groups_nor], axis=0)

print(f"\nTotal windows: {len(X)}  (accident={y.sum()}, normal={len(y)-y.sum()})")
print(f"Total unique trial groups: {len(set(groups))}")


Accident windows: 7093  from 7 trials
Normal windows  : 33400  from 40 trials

Total windows: 40493  (accident=7093, normal=33400)
Total unique trial groups: 47


In [21]:
mean = X.mean(axis=(0,1), keepdims=True)
std  = X.std(axis=(0,1),  keepdims=True) + 1e-8
X_n  = (X - mean) / std

print(f"\nNorm mean: {mean.squeeze().tolist()}")
print(f"Norm std : {std.squeeze().tolist()}")


Norm mean: [0.06207677721977234, 0.019389599561691284, 3.072171926498413, -0.038939863443374634, -0.5478214025497437, 0.21976105868816376]
Norm std : [0.9281962513923645, 0.11619482934474945, 3.921076774597168, 5.132999897003174, 4.932448863983154, 5.554668426513672]


In [22]:
# ── Cell 8-FIX: Diagnose group counts, then do a STRATIFIED group split ───────

# First, let's see exactly what we're working with
unique_groups = np.unique(groups)
acc_groups = np.unique(groups[y == 1])
nor_groups = np.unique(groups[y == 0])

print(f"Total unique groups       : {len(unique_groups)}")
print(f"Accident trial groups     : {len(acc_groups)}  -> {list(acc_groups)}")
print(f"Normal pseudo-trial groups: {len(nor_groups)}")

# The problem: GroupShuffleSplit doesn't stratify by class, so with few
# accident groups, random luck can put them ALL on one side.
# Fix: manually ensure both train and test get a fair share of accident trials.

from sklearn.model_selection import StratifiedGroupKFold

# Use StratifiedGroupKFold with enough folds that one fold ~= 15% test size,
# and pick a fold that actually balances classes
N_SPLITS_FOR_HOLDOUT = 6  # gives ~17% holdout, close to our target 15%

sgkf_holdout = StratifiedGroupKFold(n_splits=N_SPLITS_FOR_HOLDOUT,
                                     shuffle=True, random_state=42)

# Find a split where the test fold actually contains accident windows
best_split = None
for split_num, (tv_idx, test_idx) in enumerate(
        sgkf_holdout.split(X_n, y, groups)):
    n_acc_in_test = y[test_idx].sum()
    n_acc_groups_in_test = len(set(groups[test_idx]) & set(acc_groups))
    print(f"Split {split_num}: test_size={len(test_idx)}, "
          f"accident_windows_in_test={n_acc_in_test}, "
          f"accident_trials_in_test={n_acc_groups_in_test}")
    if n_acc_in_test > 0 and best_split is None:
        best_split = (tv_idx, test_idx)

if best_split is None:
    raise ValueError(
        "No split found with accident samples in test set. "
        "You likely have too few accident trial groups for group-based "
        "splitting to work reliably. See note below."
    )

train_val_idx, test_idx = best_split

X_trainval, y_trainval, groups_trainval = X_n[train_val_idx], y[train_val_idx], groups[train_val_idx]
X_test,     y_test,     groups_test     = X_n[test_idx],      y[test_idx],      groups[test_idx]

print(f"\nFinal split:")
print(f"Train+Val windows: {len(X_trainval)}  (accident={y_trainval.sum()})  "
      f"from {len(set(groups_trainval))} trials")
print(f"Test windows      : {len(X_test)}  (accident={y_test.sum()})  "
      f"from {len(set(groups_test))} trials")
print(f"Test accident trials: {set(groups_test) & set(acc_groups)}")

# Verify no leakage
assert len(set(groups_trainval) & set(groups_test)) == 0, "LEAKAGE DETECTED!"
print("\nNo group leakage between train+val and test confirmed.")

np.save('X_trainval.npy',      X_trainval)
np.save('y_trainval.npy',      y_trainval)
np.save('groups_trainval.npy', groups_trainval)
np.save('X_test.npy',          X_test)
np.save('y_test.npy',          y_test)

print("\nSaved. Proceed to k-fold training cell.")

Total unique groups       : 47
Accident trial groups     : 7  -> [np.str_('Fall Like manoeuvre 1'), np.str_('Fall Like manoeuvre 2'), np.str_('Fall Like manoeuvre 3'), np.str_('Fall in a curve'), np.str_('Fall in the roundabout'), np.str_('Fall on a slippery straight road section'), np.str_('Fall with leaning of the motorcycle')]
Normal pseudo-trial groups: 40
Split 0: test_size=5749, accident_windows_in_test=739, accident_trials_in_test=1
Split 1: test_size=5672, accident_windows_in_test=662, accident_trials_in_test=1
Split 2: test_size=6628, accident_windows_in_test=783, accident_trials_in_test=1
Split 3: test_size=7148, accident_windows_in_test=2138, accident_trials_in_test=2
Split 4: test_size=8168, accident_windows_in_test=1488, accident_trials_in_test=1
Split 5: test_size=7128, accident_windows_in_test=1283, accident_trials_in_test=1

Final split:
Train+Val windows: 34744  (accident=6354)  from 40 trials
Test windows      : 5749  (accident=739)  from 7 trials
Test accident trials

In [23]:
WINDOW_SIZE = 200
N_CHANNELS  = 6

def build_model():
    inp = keras.Input(shape=(WINDOW_SIZE, N_CHANNELS), name="imu_input")

    x = keras.layers.Conv1D(32, 7, padding='same', activation='relu')(inp)
    x = keras.layers.MaxPooling1D(2)(x)
    x = keras.layers.Dropout(0.3)(x)

    x = keras.layers.Conv1D(64, 5, padding='same', activation='relu')(x)
    x = keras.layers.MaxPooling1D(2)(x)
    x = keras.layers.Dropout(0.3)(x)

    x = keras.layers.Conv1D(64, 3, padding='same', activation='relu')(x)
    x = keras.layers.MaxPooling1D(2)(x)
    x = keras.layers.Dropout(0.4)(x)

    x = keras.layers.GlobalAveragePooling1D()(x)
    out = keras.layers.Dense(1, activation='sigmoid', name="accident_prob")(x)

    return keras.Model(inp, out, name="AccidentDetector_1DCNN")

In [24]:
from scipy.interpolate import CubicSpline

def add_gaussian_noise(x, sigma=0.05):
    return x + np.random.normal(0, sigma, x.shape).astype(np.float32)

def scale_amplitude(x, sigma=0.15):
    scale = np.random.normal(1.0, sigma, (1, x.shape[1])).astype(np.float32)
    return x * scale

def time_warp(x, sigma=0.2, knots=4):
    orig_steps  = np.arange(len(x))
    warp_steps  = np.sort(np.random.choice(orig_steps[1:-1], knots, replace=False))
    warp_steps  = np.concatenate([[0], warp_steps, [len(x)-1]])
    warp_values = warp_steps + np.random.normal(0, sigma*len(x), len(warp_steps))
    warp_values[0] = 0; warp_values[-1] = len(x)-1
    warp_values = np.sort(warp_values)
    warper = CubicSpline(warp_steps, warp_values)
    new_steps = np.clip(warper(orig_steps), 0, len(x)-1)
    return np.array([np.interp(new_steps, orig_steps, x[:,ch])
                     for ch in range(x.shape[1])], dtype=np.float32).T

def axis_flip(x):
    out = x.copy()
    for i in range(3):
        if np.random.rand() < 0.3: out[:, i] *= -1
    for i in range(3, 6):
        if np.random.rand() < 0.3: out[:, i] *= -1
    return out

def augment_one(x):
    if np.random.rand() < 0.8: x = add_gaussian_noise(x)
    if np.random.rand() < 0.6: x = scale_amplitude(x)
    if np.random.rand() < 0.5: x = time_warp(x)
    if np.random.rand() < 0.4: x = axis_flip(x)
    return x

def augment_fold(X_fold, y_fold, factor=5):
    """Augment only the minority (accident) class within this fold."""
    acc_idx = np.where(y_fold == 1)[0]
    aug_X, aug_y = [], []
    for idx in acc_idx:
        for _ in range(factor):
            aug_X.append(augment_one(X_fold[idx]))
            aug_y.append(1)
    if len(aug_X) == 0:
        return X_fold, y_fold
    X_aug = np.concatenate([X_fold, np.array(aug_X, dtype=np.float32)], axis=0)
    y_aug = np.concatenate([y_fold, np.array(aug_y, dtype=np.int8)], axis=0)
    perm = np.random.permutation(len(X_aug))
    return X_aug[perm], y_aug[perm]

In [25]:
# ── Cell 11-ALT: Leave-One-Accident-Trial-Out Cross-Validation ────────────────
# With only 7 accident trials, this gives the cleanest picture: each fold
# holds out exactly ONE accident maneuver type entirely, so we directly see
# "if the model has never seen THIS type of fall, can it still catch it?"
#
# Normal data folds are split evenly across the 7 folds for balance.

X_trainval      = np.load('X_trainval.npy')
y_trainval      = np.load('y_trainval.npy')
groups_trainval = np.load('groups_trainval.npy', allow_pickle=True)

acc_trial_names = sorted(set(groups_trainval[y_trainval == 1]))
nor_trial_names = sorted(set(groups_trainval[y_trainval == 0]))

print(f"Accident trials available for LOTO-CV: {len(acc_trial_names)}")
for t in acc_trial_names:
    print(f"  - {t}")
print(f"Normal pseudo-trials available: {len(nor_trial_names)}")

# Distribute normal trials evenly across folds (one chunk of normal trials
# held out alongside each accident trial, so val sets stay balanced)
np.random.seed(42)
shuffled_normal = np.random.permutation(nor_trial_names)
normal_folds = np.array_split(shuffled_normal, len(acc_trial_names))

loto_results = []

for fold_idx, held_out_acc_trial in enumerate(acc_trial_names):
    print(f"\n{'='*60}")
    print(f"  LOTO FOLD {fold_idx + 1}/{len(acc_trial_names)}: "
          f"holding out '{held_out_acc_trial}'")
    print(f"{'='*60}")

    held_out_normal_trials = set(normal_folds[fold_idx])
    val_groups   = {held_out_acc_trial} | held_out_normal_trials
    train_groups = (set(acc_trial_names) | set(nor_trial_names)) - val_groups

    train_mask = np.isin(groups_trainval, list(train_groups))
    val_mask   = np.isin(groups_trainval, list(val_groups))

    X_train_fold, y_train_fold = X_trainval[train_mask], y_trainval[train_mask]
    X_val_fold,   y_val_fold   = X_trainval[val_mask],   y_trainval[val_mask]

    print(f"Train: {len(X_train_fold)} windows ({y_train_fold.sum()} accident) "
          f"from {len(train_groups)} trials")
    print(f"Val  : {len(X_val_fold)} windows ({y_val_fold.sum()} accident) "
          f"-- held-out trial: '{held_out_acc_trial}' + "
          f"{len(held_out_normal_trials)} normal chunks")

    # Augment training data only
    X_train_aug, y_train_aug = augment_fold(X_train_fold, y_train_fold, factor=5)

    n_pos = y_train_aug.sum()
    n_neg = len(y_train_aug) - n_pos
    class_weight = {0: 1.0, 1: float(n_neg / max(n_pos, 1))}

    model = build_model()
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=[keras.metrics.BinaryAccuracy(name='acc'),
                 keras.metrics.Recall(name='recall'),
                 keras.metrics.Precision(name='precision')]
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='loss', patience=10, restore_best_weights=True, verbose=0
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='loss', factor=0.5, patience=6, min_lr=1e-5, verbose=0
        ),
    ]

    model.fit(
        X_train_aug, y_train_aug,
        epochs=50,
        batch_size=32,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=0,
    )

    y_prob = model.predict(X_val_fold, verbose=0).squeeze()
    y_pred = (y_prob >= 0.5).astype(int)

    from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

    # Recall specifically on the held-out accident trial's windows
    acc_only_mask = groups_trainval[val_mask] == held_out_acc_trial
    acc_trial_recall = recall_score(
        y_val_fold[acc_only_mask], y_pred[acc_only_mask], zero_division=0
    ) if acc_only_mask.sum() > 0 else float('nan')

    fold_metrics = {
        'held_out_trial': held_out_acc_trial,
        'val_windows':    len(X_val_fold),
        'accuracy':       accuracy_score(y_val_fold, y_pred),
        'recall_overall': recall_score(y_val_fold, y_pred, zero_division=0),
        'recall_on_held_out_trial_only': acc_trial_recall,
        'precision':      precision_score(y_val_fold, y_pred, zero_division=0),
        'f1':             f1_score(y_val_fold, y_pred, zero_division=0),
    }
    loto_results.append(fold_metrics)

    print(f"\nResults -- recall on '{held_out_acc_trial}' windows specifically: "
          f"{acc_trial_recall:.4f}")
    print(f"Overall fold accuracy: {fold_metrics['accuracy']:.4f}")

Accident trials available for LOTO-CV: 6
  - Fall Like manoeuvre 1
  - Fall Like manoeuvre 2
  - Fall Like manoeuvre 3
  - Fall in a curve
  - Fall in the roundabout
  - Fall with leaning of the motorcycle
Normal pseudo-trials available: 34

  LOTO FOLD 1/6: holding out 'Fall Like manoeuvre 1'
Train: 28411 windows (5031 accident) from 33 trials
Val  : 6333 windows (1323 accident) -- held-out trial: 'Fall Like manoeuvre 1' + 6 normal chunks

Results -- recall on 'Fall Like manoeuvre 1' windows specifically: 0.9924
Overall fold accuracy: 0.8258

  LOTO FOLD 2/6: holding out 'Fall Like manoeuvre 2'
Train: 28246 windows (4866 accident) from 33 trials
Val  : 6498 windows (1488 accident) -- held-out trial: 'Fall Like manoeuvre 2' + 6 normal chunks

Results -- recall on 'Fall Like manoeuvre 2' windows specifically: 0.9442
Overall fold accuracy: 0.9872

  LOTO FOLD 3/6: holding out 'Fall Like manoeuvre 3'
Train: 28451 windows (5071 accident) from 33 trials
Val  : 6293 windows (1283 accident) -

In [26]:
loto_df = pd.DataFrame(loto_results)
print(f"\n{'='*70}")
print("  LEAVE-ONE-ACCIDENT-TRIAL-OUT CROSS-VALIDATION SUMMARY")
print(f"{'='*70}")
print(loto_df.to_string(index=False))

print(f"\nThis table tells you, per fall TYPE, whether the model generalizes")
print(f"to a maneuver it never saw during training. A low recall_on_held_out_trial")
print(f"for a specific row means that maneuver type is hard for the model to")
print(f"detect when unseen -- worth discussing in your report.")

print(f"\nMean recall on held-out trial type: "
      f"{loto_df['recall_on_held_out_trial_only'].mean():.4f} +/- "
      f"{loto_df['recall_on_held_out_trial_only'].std():.4f}")
print(f"Mean overall fold accuracy: "
      f"{loto_df['accuracy'].mean():.4f} +/- {loto_df['accuracy'].std():.4f}")


  LEAVE-ONE-ACCIDENT-TRIAL-OUT CROSS-VALIDATION SUMMARY
                     held_out_trial  val_windows  accuracy  recall_overall  recall_on_held_out_trial_only  precision       f1
              Fall Like manoeuvre 1         6333  0.825833        0.992441                       0.992441   0.545719 0.704210
              Fall Like manoeuvre 2         6498  0.987227        0.944220                       0.944220   1.000000 0.971310
              Fall Like manoeuvre 3         6293  0.991419        0.957911                       0.957911   1.000000 0.978503
                    Fall in a curve         5793  0.971690        0.790549                       0.790549   1.000000 0.883024
             Fall in the roundabout         4837  0.915237        0.989426                       0.989426   0.619093 0.761628
Fall with leaning of the motorcycle         4990  0.977154        0.920245                       0.920245   0.938673 0.929368

This table tells you, per fall TYPE, whether the model gener

In [27]:
# Check why only 6 folds ran instead of 7
acc_trial_names_check = sorted(set(groups_trainval[y_trainval == 1]))
print(f"Accident trials actually in groups_trainval: {len(acc_trial_names_check)}")
for t in acc_trial_names_check:
    print(f"  - {t}")

print(f"\nOriginal 7 accident trials were:")
original_7 = ['Fall Like manoeuvre 1', 'Fall Like manoeuvre 2', 'Fall Like manoeuvre 3',
              'Fall in a curve', 'Fall in the roundabout',
              'Fall on a slippery straight road section',
              'Fall with leaning of the motorcycle']
for t in original_7:
    present = t in acc_trial_names_check
    print(f"  {'✓' if present else '✗ MISSING'}  {t}")


Accident trials actually in groups_trainval: 6
  - Fall Like manoeuvre 1
  - Fall Like manoeuvre 2
  - Fall Like manoeuvre 3
  - Fall in a curve
  - Fall in the roundabout
  - Fall with leaning of the motorcycle

Original 7 accident trials were:
  ✓  Fall Like manoeuvre 1
  ✓  Fall Like manoeuvre 2
  ✓  Fall Like manoeuvre 3
  ✓  Fall in a curve
  ✓  Fall in the roundabout
  ✗ MISSING  Fall on a slippery straight road section
  ✓  Fall with leaning of the motorcycle


In [28]:
X_trainval      = np.load('X_trainval.npy')
y_trainval      = np.load('y_trainval.npy')
X_test          = np.load('X_test.npy')
y_test          = np.load('y_test.npy')

X_trainval_aug, y_trainval_aug = augment_fold(X_trainval, y_trainval, factor=5)
print(f"Final training set: {len(X_trainval_aug)} windows "
      f"({y_trainval_aug.sum()} accident)")

n_pos = y_trainval_aug.sum()
n_neg = len(y_trainval_aug) - n_pos
class_weight = {0: 1.0, 1: float(n_neg / max(n_pos, 1))}

final_model = build_model()
final_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=[keras.metrics.BinaryAccuracy(name='acc'),
             keras.metrics.Recall(name='recall'),
             keras.metrics.Precision(name='precision')]
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        'final_model_best.keras', monitor='recall', mode='max',
        save_best_only=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='loss', factor=0.5, patience=6, min_lr=1e-5, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='loss', patience=12, restore_best_weights=True, verbose=1
    ),
]

history_final = final_model.fit(
    X_trainval_aug, y_trainval_aug,
    epochs=50,
    batch_size=32,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)


Final training set: 66514 windows (38124 accident)
Epoch 1/50
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - acc: 0.9420 - loss: 0.1518 - precision: 0.9314 - recall: 0.9705
Epoch 1: recall improved from None to 0.98072, saving model to final_model_best.keras

Epoch 1: finished saving model to final_model_best.keras
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - acc: 0.9605 - loss: 0.1090 - precision: 0.9518 - recall: 0.9807 - learning_rate: 0.0010
Epoch 2/50
2066/2079 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - acc: 0.9726 - loss: 0.0818 - precision: 0.9658 - recall: 0.9870
Epoch 2: recall improved from 0.98072 to 0.98817, saving model to final_model_best.keras

Epoch 2: finished saving model to final_model_best.keras
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - acc: 0.9733 - loss: 0.0809 - precision: 0.9660 - recall: 0.9882 - learning_rate: 0.0010
Epoch 3/50
2069/2079 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - acc: 0.9749 - loss: 0.0782 - precision: 0.9664 - recall: 0.9905
Epoch 3: recall improved from 

In [29]:
final_model = keras.models.load_model('final_model_best.keras')

y_prob_test = final_model.predict(X_test, verbose=0).squeeze()
y_pred_test = (y_prob_test >= 0.5).astype(int)

print(f"\n{'='*60}")
print("  FINAL HOLDOUT TEST SET EVALUATION")
print(f"  (This is the most honest estimate of real-world performance)")
print(f"{'='*60}")
print(classification_report(y_test, y_pred_test, target_names=["Normal", "Accident"]))

cm = confusion_matrix(y_test, y_pred_test)
tn, fp, fn, tp = cm.ravel()
print(f"Confusion matrix:")
print(f"  TN={tn}  FP={fp}")
print(f"  FN={fn}  TP={tp}")
print(f"\nFalse alarm rate (FPR): {fp/(fp+tn):.4f}")
print(f"Miss rate (FNR)       : {fn/(fn+tp):.4f}")
print(f"ROC-AUC               : {roc_auc_score(y_test, y_prob_test):.4f}")

# Compare to k-fold mean — should be in a similar ballpark
from sklearn.metrics import recall_score
print(f"\nFor comparison, 5-fold CV mean recall was: "
      f"{results_df['recall'].mean():.4f} ± {results_df['recall'].std():.4f}")
print(f"Test set recall: {recall_score(y_test, y_pred_test):.4f}")
print("\nIf test recall falls within the CV range, the result is trustworthy.")
print("If test recall is far outside that range, something is off — investigate.")


  FINAL HOLDOUT TEST SET EVALUATION
  (This is the most honest estimate of real-world performance)
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      5010
    Accident       1.00      0.99      1.00       739

    accuracy                           1.00      5749
   macro avg       1.00      1.00      1.00      5749
weighted avg       1.00      1.00      1.00      5749

Confusion matrix:
  TN=5010  FP=0
  FN=7  TP=732

False alarm rate (FPR): 0.0000
Miss rate (FNR)       : 0.0095
ROC-AUC               : 1.0000

For comparison, 5-fold CV mean recall was: 0.9822 ± 0.0393
Test set recall: 0.9905

If test recall falls within the CV range, the result is trustworthy.
If test recall is far outside that range, something is off — investigate.


In [33]:
from google.colab import files
import tensorflow as tf

# Define MODELS_DIR, assuming it should be a subdirectory within DRIVE_ROOT
# DRIVE_ROOT is defined in cell wBWnCAoBH2fE and Path is imported in cell tZIjBXMMBFun.
MODELS_DIR = DRIVE_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True) # Create the directory if it doesn't exist

# Load the best Keras model
final_model = tf.keras.models.load_model('final_model_best.keras')

# Convert to TensorFlow Lite model with integer quantization
converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Create a representative dataset for int8 quantization
# Using a small subset of X_trainval for calibration
X_trainval      = np.load('X_trainval.npy')
def representative_dataset_gen():
    for i in range(100): # Use 100 samples for calibration
        yield [X_trainval[i:i+1].astype(np.float32)]

converter.representative_dataset = representative_dataset_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_int8 = converter.convert()

# Define the path to save the quantized model
model_int8_path = MODELS_DIR / "model_int8.h"

# Save the quantized model
with open(model_int8_path, "wb") as f:
    f.write(tflite_model_int8)

print(f"Quantized model saved to: {model_int8_path}")

# Download the saved model
files.download(str(model_int8_path))

Saved artifact at '/tmp/tmp33mgb9z0'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 200, 6), dtype=tf.float32, name='imu_input')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136804428399440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136804428401744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136804428398672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136804428402704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136804428399632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136804428404816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136804428401552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136804428405392: TensorSpec(shape=(), dtype=tf.resource, name=None)
Quantized model saved to: /content/drive/MyDrive/Colab Notebooks/models/model_int8.h


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
import numpy as np
from pathlib import Path

# Define PROCESSED_DIR to point to the root content directory
PROCESSED_DIR = Path('/content/')

mean = np.load(str(PROCESSED_DIR / "norm_mean.npy"))
std  = np.load(str(PROCESSED_DIR / "norm_std.npy"))
print("NORM_MEAN:", mean.tolist())
print("NORM_STD: ", std.tolist())

NORM_MEAN: [0.06207677721977234, 0.019389599561691284, 3.072171926498413, -0.038939863443374634, -0.5478214025497437, 0.21976105868816376]
NORM_STD:  [0.9281962513923645, 0.11619482934474945, 3.921076774597168, 5.132999897003174, 4.932448863983154, 5.554668426513672]
